In [ ]:
import sys
from pathlib import Path

import duckdb
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / ".git").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import DB_PATH, FIGURES_DIR, ensure_dirs

ensure_dirs()

from src.viz import apply_chinese, get_font_prop

font_prop = get_font_prop()
plt.rcParams["axes.unicode_minus"] = False

con = duckdb.connect(str(DB_PATH), read_only=True)


In [ ]:
repurchase = con.sql("""
WITH buyer_stats AS (
    SELECT user_id, COUNT(*) AS buy_cnt
    FROM user_behavior
    WHERE behavior_type = '购买'
    GROUP BY user_id
)
SELECT buy_cnt, COUNT(*) AS user_cnt
FROM buyer_stats
GROUP BY buy_cnt
ORDER BY buy_cnt
LIMIT 10
""").df()
repurchase

In [ ]:
fig, ax = plt.subplots(figsize=(8,4))
sns.barplot(data=repurchase,x = "buy_cnt", y = "user_cnt", ax=ax, color="#dd8452")
for i, v in enumerate(repurchase['user_cnt']):
    ax.text(i,v,f"{v:,}", ha="center", va="bottom", fontsize=9)
ax.set_title("用户购买次数分布")
ax.set_xlabel("购买次数")
ax.set_ylabel("用户数")
apply_chinese(ax, font_prop)

plt.savefig(FIGURES_DIR/"repurchase_dist.png")
plt.show()